# gatenet v3 retrain -- VQ1 ONLY (no hand labels)

Identical to `gatenet_colab_v3.ipynb` except that it trains on `autolabels_vq1_v3.json`
alone and needs no `vq2_frames.zip`. Purpose: isolate ONE variable. v3 changed the labels
(gate 1/2/4 pose refits, behind-gate removals, rate weighting); the hand labels are a
SECOND change. Running them together would repeat the mistake that made the last
regression un-diagnosable. Run this first, then the merged notebook.

Upload to Drive: updated `gatenet.py`, `packcrops.py`, `autolabels_vq1_v3.json` into
`MyDrive/vqual2/perception/`. `vq1_frames.zip` and `gatenet_runs/block/best.pt` are
already there. Nothing else.

---

# gatenet v3 on Colab -- the single audited retrain

Same machinery as `gatenet_colab.ipynb` (which stays untouched as the baseline record):
imports `gatenet.py`, rebuilds the crop cache with `packcrops.py`, calls `gatenet.train()`.
Three things differ, and only these three:

1. **Labels**: `labels_merged_v3.json` -- the audited `autolabels_vq1_v3.json` (gate 1/2/4
   pose refits, behind-gate negatives removed, per-instance `body_rate`) merged with
   Claire's VQ2 hand labels via `mergelabels.py` (`src: "hand"`).
2. **`--weight-mode rate`**: auto-label error is rate-dependent (NOTES.md 2026-08-01:
   ~1 px below 0.2 rad/s, 3.4 px above 1 rad/s -- attitude aliasing), so fast-rotation
   auto instances are down-weighted 1.0 -> 0.30 on a raised cosine between 0.5 and
   2 rad/s. Hand labels are drawn on the frame, carry no aliasing, and keep weight 1.0.
3. **A final CROSS-EVAL cell**: the new best checkpoint AND the current production model
   `gatenet_runs/block/best.pt` scored on the IDENTICAL new val split, with per-rate and
   per-src breakdowns. The last label-change retrain regressed (1.60 vs 1.18 px) and the
   cause was only settled by exactly this comparison (`crosseval_colab.py`), so this run
   bakes it in: the accept/reject decision is one table read, not an argument.

Hyperparameters are the colab baseline's, unchanged: batch 256, lr 3e-4*(batch/64),
200 epochs, seed 0, block split. One training variable changes (the weighting);
everything else is held so a difference in the table means what it says.

**Negatives** (`labelfix_negatives_v3.json`) stay unused: there is no confidence head in
this run, deliberately.

---

## WHAT TO UPLOAD, AND WHERE

Locally first (all three commands are in TRAINING.md's v3 runbook):

```
python3 pilot/perception/mergelabels.py --hand <labels_gates.json from Claire> \
        --auto pilot/perception/autolabels_vq1_v3.json --drop-unsure \
        --out pilot/perception/labels_merged_v3.json
python3 pilot/perception/_mkframezip_vq2.py     # -> vq2_frames.zip (hand-labelled frames)
```

Then on Drive, in **`MyDrive/vqual2/`** (same folder the baseline used):

```
MyDrive/vqual2/
  perception/
    gatenet.py               <- CURRENT copy (has --weight-mode / --labels)
    packcrops.py             <- CURRENT copy (index carries body_rate/src)
    autolabel.py             <- unchanged
    label.py                 <- unchanged
    labels_merged_v3.json    <- the merge output above
  vq1_frames.zip             <- already there from the baseline run
  vq2_frames.zip             <- NEW, from _mkframezip_vq2.py
  gatenet_runs/
    block/best.pt            <- already there; the PRODUCTION model the cross-eval needs
```

Upload zips with the Drive desktop client or drive.google.com, not from a Colab cell.
Frames beyond the two zips are not needed; neither is a starting checkpoint.

## 1. What did Colab actually give us?

Colab varies by the hour: T4 / L4 / A100, 2-12 vCPU, 12-85 GB RAM. Batch size, worker
count and whether the cache fits in RAM depend on it, so measure rather than assume.

In [ ]:
import os, subprocess, sys, time, shutil, json

print("--- GPU " + "-" * 60)
try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip())
except Exception as e:
    print("no nvidia-smi:", e)

import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  {p.name}  {p.total_memory/2**30:.1f} GiB  sm_{p.major}{p.minor}  "
          f"{p.multi_processor_count} SMs")

print("--- CPU / RAM " + "-" * 53)
print("vCPU (os.cpu_count):", os.cpu_count())
mem = {}
for line in open("/proc/meminfo"):
    k, v = line.split(":", 1)
    mem[k] = v.strip()
print("MemTotal:", mem.get("MemTotal"), "| MemAvailable:", mem.get("MemAvailable"))
print("--- DISK (/content) " + "-" * 48)
t, u, f = shutil.disk_usage("/content")
print(f"total {t/2**30:.0f} GiB, free {f/2**30:.0f} GiB")
assert f > 8 * 2**30, "not enough local disk for the ~4.5 GB cache + checkpoints"

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE       = "/content/drive/MyDrive/vqual2"   # <- the one path to change
CODE_DRIVE  = f"{DRIVE}/perception"
LABELS_NAME = "autolabels_vq1_v3.json"          # AUDITED VQ1 labels, no hand labels in this run
VQ1_ZIP     = f"{DRIVE}/vq1_frames.zip"         # 247 MB, the labelled VQ1 frames
VQ2_ZIP     = None                              # this variant trains on VQ1 only
RUNS_DRIVE  = f"{DRIVE}/gatenet_runs"
OLD_BEST    = f"{RUNS_DRIVE}/block/best.pt"     # PRODUCTION model; the cross-eval baseline

# Everything below lives on Colab's local NVMe, never on the Drive FUSE mount.
CODE_LOCAL  = "/content/code"
SESS_LOCAL  = "/content/sessions"
LOCAL_CACHE = "/content/crop_cache"

for p_ in (DRIVE, CODE_DRIVE):
    assert os.path.isdir(p_), f"missing on Drive: {p_}  (see the upload list at the top)"
for f_ in (VQ1_ZIP, f"{CODE_DRIVE}/{LABELS_NAME}"):
    assert os.path.isfile(f_), f"missing on Drive: {f_}  (see the upload list at the top)"
# checked NOW, not after 3 h of training: the whole point of this run is the comparison
assert os.path.isfile(OLD_BEST), f"missing on Drive: {OLD_BEST} -- cross-eval impossible"
os.makedirs(RUNS_DRIVE, exist_ok=True)
print("Drive OK.", LABELS_NAME,
      f"{os.path.getsize(f'{CODE_DRIVE}/{LABELS_NAME}')/1e6:.1f} MB;",
      f"vq1 zip {os.path.getsize(VQ1_ZIP)/1e6:.0f} MB (no VQ2 zip in this variant)")

## 3. Unzip the frames and REBUILD the cache here

Same pattern as the baseline notebook: upload the small thing (frames), rebuild the big
thing (tiles) on local NVMe. The only difference is the second zip and
`--labels labels_merged_v3.json`, which also becomes the cache's staleness fingerprint
(`labels_sha256`), so a stale cache against the wrong label file is a hard error later,
not a silent wrong table.

In [ ]:
import zipfile

# code off Drive onto local disk, so ROOT resolves locally and imports are not FUSE reads
os.makedirs(f"{CODE_LOCAL}/pilot", exist_ok=True)
shutil.rmtree(f"{CODE_LOCAL}/pilot/perception", ignore_errors=True)
shutil.copytree(CODE_DRIVE, f"{CODE_LOCAL}/pilot/perception")
LABELS_LOCAL = f"{CODE_LOCAL}/pilot/perception/{LABELS_NAME}"

# frames: both zips into the same sessions tree
if not os.path.isdir(SESS_LOCAL):
    os.makedirs(SESS_LOCAL, exist_ok=True)
    for zp in (VQ1_ZIP,):
        t0 = time.time()
        with zipfile.ZipFile(zp) as z:
            n = len(z.namelist())
            z.extractall(SESS_LOCAL)
        print(f"unzipped {n} frames from {os.path.basename(zp)} in {time.time()-t0:.0f} s")
else:
    print("frames already unzipped")

link = f"{CODE_LOCAL}/pilot/sessions"
if not os.path.islink(link) and not os.path.isdir(link):
    os.symlink(SESS_LOCAL, link)

# rebuild the cache against the MERGED labels. meta.json's missing_frames MUST be 0:
# a nonzero count means hand-labelled frames did not make it into vq2_frames.zip and
# their tiles are black.
t0 = time.time()
r = subprocess.run([sys.executable, f"{CODE_LOCAL}/pilot/perception/packcrops.py",
                    "--mode", "pack", "--out", LOCAL_CACHE, "--labels", LABELS_LOCAL],
                   capture_output=True, text=True)
print(r.stdout[-2000:] or r.stderr[-2000:])
assert r.returncode == 0, "pack failed -- read the output above"
meta_ = json.load(open(f"{LOCAL_CACHE}/meta.json"))
print(f"cache rebuilt in {time.time()-t0:.0f} s")
assert meta_["missing_frames"] == 0, \
    f"{meta_['missing_frames']} frames unreadable -- vq1_frames.zip is incomplete"
print(json.dumps(meta_, indent=1))

## 4. Import gatenet unchanged and point it at the cache

`G.LABELS` is set to the merged file BEFORE `attach()`: the fingerprint check hashes
`G.LABELS`, so this is what makes a cache/label mismatch a hard error. The split stays
label-file-driven `block` exactly as before -- VQ2 hand-labelled sessions get blocked by
time like every other session, so held-out hand frames exist and the `src hand` eval row
is a real holdout, not training data re-read.

In [ ]:
sys.path.insert(0, f"{CODE_LOCAL}/pilot/perception")
import gatenet as G
import packcrops as P

G.LABELS = LABELS_LOCAL          # before attach(): the fingerprint hashes G.LABELS
items, tiles, meta = P.attach(LOCAL_CACHE)
print(f"instances: {len(items)}   tile {meta['tile_res']}x{meta['tile_res']} "
      f"scale {meta['tile_scale']}   eval_exact={meta['eval_exact']}")

import numpy as np
src_ = np.array([d['src'] for d in items])
rate_ = np.array([d['body_rate'] for d in items])
print(f"src: auto {int((src_=='auto').sum())} / hand {int((src_=='hand').sum())};   "
      f"body_rate>1 rad/s: {int((rate_>1).sum())} "
      f"({100*(rate_>1).mean():.1f}%, expect ~21.6% of the auto set)")
w_ = np.array([G.instance_weight(d) for d in items])
print(f"instance weights: mean {w_.mean():.3f}  min {w_.min():.3f}  "
      f"({int((w_<1).sum())} down-weighted; every src=hand weight is "
      f"{set(np.round(w_[src_=='hand'],3)) if (src_=='hand').any() else 'n/a'})")

tr_items, va_items = G.split_index(items, "block")
print(f"block split: train {len(tr_items)} / val {len(va_items)}   "
      f"(hand in val: {sum(1 for d in va_items if d['src']=='hand')})")

### 4b. Sanity: the tiles are pictures of gates, in the right frame

Two minutes here beats a night of training a frame bug. Green = ground-truth corners from
the target the loader will hand the net, yellow dot = corner 0 (the `min(x+y)` winding
corner). The last row samples HAND-labelled instances specifically -- they came through a
different tool (`labelui.html` -> `mergelabels.py`) and a winding or key-mapping bug there
would poison exactly the in-domain data this run exists for. If a quad is off an aperture
or the yellow dot wanders, stop.

In [ ]:
import cv2
from matplotlib import pyplot as plt

ds = P.CachedGateCrops(va_items, LOCAL_CACHE, meta, train=False)
hand_idx = [k for k, d in enumerate(va_items) if d['src'] == 'hand']
sel = list(np.linspace(0, len(ds) - 1, 8).astype(int))
sel += list(np.array(hand_idx)[np.linspace(0, len(hand_idx) - 1,
                                           min(4, len(hand_idx))).astype(int)]) \
       if hand_idx else []
cols_ = 4
rows_ = (len(sel) + cols_ - 1) // cols_
fig, ax = plt.subplots(rows_, cols_, figsize=(16, 4 * rows_))
for a in np.ravel(ax):
    a.axis("off")
for a, k in zip(np.ravel(ax), sel):
    x, y, g, _ = ds[int(k)]
    img = ((x * 0.25 + 0.45) * 255).clamp(0, 255).byte().numpy().transpose(1, 2, 0)
    img = np.ascontiguousarray(img[:, :, ::-1])          # BGR -> RGB for matplotlib
    q = (y.numpy().reshape(4, 2) + 1.0) * (G.RES / 2.0)
    cv2.polylines(img, [q.astype(np.int32).reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    cv2.circle(img, tuple(q[0].astype(int)), 4, (255, 255, 0), -1)
    d = va_items[int(k)]
    a.imshow(img)
    a.set_title(f"{d['size_px']:.0f} px {d['src']}"
                + (" CLIP" if d['clipped'] else "")
                + f"  w={G.instance_weight(d):.2f}")
plt.tight_layout(); plt.show()

## 5. Train

Baseline hyperparameters, one new flag. `gatenet.train()` writes `last.pt` + the metrics
CSV every epoch and `best.pt` on every median improvement; the epoch line ends in
`w~0.xxx`, the mean applied weight -- if that reads 1.000, the weighting is NOT active
and the run is not the run you meant to start.

Set `RESUME = True` and re-run this cell after a disconnect; resuming with a different
`EPOCHS` deliberately reshapes the cosine (documented in `gatenet.py`).

In [ ]:
import types, threading

TAG = "colab-v3-vq1"
SPLIT = "block"
EPOCHS = 200
RESUME = False           # flip to True after a disconnect and re-run this cell
BATCH = 256
WORKERS = min(8, os.cpu_count() or 2)
MAX_HOURS = 3.0
WEIGHT_MODE = "rate"     # the one training variable this run changes

# LR SCALES WITH BATCH. TRAINING.md's baseline is lr 3e-4 at batch 64. Holding 3e-4 while
# quadrupling the batch quarters the per-sample step and the run would simply learn slower --
# it would look like a worse method when it is a different schedule. Linear scaling is the
# standard correction and is applied here rather than silently inherited.
LR = 3e-4 * (BATCH / 64)

# CHECKPOINTS GO TO LOCAL DISK, NOT DRIVE. gatenet.train writes each checkpoint as `.tmp`
# then os.replace()s it into place -- an atomic rename that the Drive FUSE mount does not
# reliably implement, and it would fail at the END of epoch 1, after the slow part. So train
# against local NVMe and mirror to Drive on a timer, which also survives a disconnect.
RUNS_LOCAL = "/content/gatenet_runs"
os.makedirs(f"{RUNS_LOCAL}/{TAG}", exist_ok=True)
os.makedirs(f"{RUNS_DRIVE}/{TAG}", exist_ok=True)
G.RUNS = RUNS_LOCAL
G.REPORT = "/content/TRAINING_colab_v3.md"

_stop = threading.Event()
def _mirror(every=300):
    """Copy checkpoints + log to Drive periodically. A disconnect costs at most `every`."""
    while not _stop.wait(every):
        try:
            for n in ("best.pt", "last.pt", "log.csv"):
                s = f"{RUNS_LOCAL}/{TAG}/{n}"
                if os.path.exists(s):
                    shutil.copyfile(s, f"{RUNS_DRIVE}/{TAG}/{n}")
        except Exception as e:
            print("mirror failed (training continues):", e)
threading.Thread(target=_mirror, daemon=True).start()

if RESUME:
    for n in ("best.pt", "last.pt", "log.csv"):
        s = f"{RUNS_DRIVE}/{TAG}/{n}"
        if os.path.exists(s) and not os.path.exists(f"{RUNS_LOCAL}/{TAG}/{n}"):
            shutil.copyfile(s, f"{RUNS_LOCAL}/{TAG}/{n}")

args = types.SimpleNamespace(
    mode="train", split=SPLIT, tag=TAG, epochs=EPOCHS, batch=BATCH, lr=LR,
    width=1.0, workers=WORKERS, max_hours=MAX_HOURS, resume=RESUME, hflip=False,
    ckpt="best", cpu=False, res=G.RES, seed=0, weight_mode=WEIGHT_MODE)

torch.manual_seed(args.seed); np.random.seed(args.seed)
cv2.setNumThreads(0); torch.backends.cudnn.benchmark = True
print(f"batch {BATCH}  lr {LR:.2e}  workers {WORKERS}  weight-mode {WEIGHT_MODE}  "
      f"-> {RUNS_LOCAL}, mirrored to Drive")

t0 = time.time()
rows = G.train(args)
_stop.set()
for n in ("best.pt", "last.pt", "log.csv"):
    s = f"{RUNS_LOCAL}/{TAG}/{n}"
    if os.path.exists(s):
        shutil.copyfile(s, f"{RUNS_DRIVE}/{TAG}/{n}")
if os.path.exists(G.REPORT):
    shutil.copyfile(G.REPORT, f"{DRIVE}/TRAINING_colab_v3.md")
print(f"\nwall clock {(time.time()-t0)/60:.1f} min; checkpoints mirrored to Drive")

## 6. The new model's report

Re-running the eval from the best checkpoint confirms the checkpoint on Drive is the one
that produced the numbers, and prints the new per-rate / per-src rows. Do NOT compare
these against TRAINING.md's 0.89 px yet -- the label file changed, so the val set
changed. The comparison that decides anything is the next cell.

In [ ]:
from torch.utils.data import DataLoader

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = G.GateNet(1.0).to(dev)
ck = torch.load(f"{RUNS_DRIVE}/{TAG}/best.pt", map_location=dev, weights_only=False)
m.load_state_dict(ck["model"]); m.eval()
print(f"best.pt: epoch {ck['epoch']}, best val corner med {ck['best']:.3f} px, "
      f"{ck['nparam']:,} params")

va = DataLoader(P.CachedGateCrops(va_items, LOCAL_CACHE, meta, train=False),
                batch_size=256, shuffle=False, num_workers=WORKERS)
res_new = G.evaluate(m, va, dev, va_items)
rows_new = G.breakdown(res_new, va_items)
print()
print(G.fmt_rows(rows_new))

## 7. CROSS-EVAL: new checkpoint vs production `block/best.pt`, same val split

The decision cell. The filtered-v1 retrain looked like a regression (1.60 px vs the 1.18
the old model scored on the SAME new set) and only this comparison settled it, so it is
not optional here. Both models, identical loader, identical instances; the tables differ
only by checkpoint.

**Accept/reject rule (from TRAINING.md's v3 runbook):** the new model must beat
`block/best.pt` on the **ALL** row (corner med) AND not regress the **clipped** row.
Otherwise production stays `block/best.pt`, and the delta table below is the diagnosis to
bring home. The old model's own `best` number in its checkpoint is from the OLD val set
and is printed for identification only -- never compare across val sets.

In [ ]:
old = G.GateNet(1.0).to(dev)
ck_old = torch.load(OLD_BEST, map_location=dev, weights_only=False)
old.load_state_dict(ck_old["model"]); old.eval()
print(f"OLD (production): {OLD_BEST}")
print(f"  epoch {ck_old['epoch']}, its own best {ck_old['best']:.3f} px "
      f"(old val set -- identification only)")
print(f"NEW: {RUNS_DRIVE}/{TAG}/best.pt   |   shared val: {len(va_items)} instances, "
      f"labels {LABELS_NAME}, block split\n")

res_old = G.evaluate(old, va, dev, va_items)
rows_old = G.breakdown(res_old, va_items)

print("### OLD (production block/best.pt) on the NEW val split")
print(G.fmt_rows(rows_old))
print()
print("### NEW (colab-v3 best.pt) on the SAME split")
print(G.fmt_rows(rows_new))

print("\n### decision table: corner med px, NEW - OLD (negative = new is better)")
print("| group | n | old | new | delta |")
print("|---|---:|---:|---:|---:|")
by_old = {r["group"]: r for r in rows_old}
verdict = {}
for r in rows_new:
    o = by_old.get(r["group"])
    if not o:
        continue
    d = r["corner_med"] - o["corner_med"]
    verdict[r["group"]] = d
    print(f"| {r['group']} | {r['n']} | {o['corner_med']:.2f} | "
          f"{r['corner_med']:.2f} | {d:+.2f} |")

ok_all = verdict.get("ALL", 1) < 0
ok_clip = verdict.get("clipped", 1) <= 0
print(f"\nALL row: new {'beats' if ok_all else 'DOES NOT beat'} old  |  "
      f"clipped: {'no regression' if ok_clip else 'REGRESSED'}")
print("=> ACCEPT: colab-v3 becomes production" if ok_all and ok_clip else
      "=> REJECT: production stays gatenet_runs/block/best.pt")

## 8. Bring the result home

Checkpoints are on Drive. Copy `gatenet_runs/colab-v3/` back into
`pilot/perception/gatenet_runs/` on the laptop, and paste BOTH cross-eval tables plus the
decision table into `TRAINING.md` under a run heading that names Colab and the GPU from
cell 1 -- the hardware is part of the measurement. If REJECT, the deltas by rate/src rows
are the first diagnostic: label error and net error are finally separated there.

In [ ]:
print("on Drive:", RUNS_DRIVE + "/" + TAG)
for f in sorted(os.listdir(f"{RUNS_DRIVE}/{TAG}")):
    print(f"  {f}  {os.path.getsize(f'{RUNS_DRIVE}/{TAG}/{f}')/1e6:.1f} MB")
print("\nper-epoch times (s) from log.csv:")
import csv as _csv
rows_ = list(_csv.DictReader(open(f"{RUNS_DRIVE}/{TAG}/log.csv")))
secs = [float(r["secs"]) for r in rows_]
if secs:
    print(f"  n={len(secs)}  median {sorted(secs)[len(secs)//2]:.1f}  "
          f"min {min(secs):.1f}  max {max(secs):.1f}")